In [ ]:
# ── Cell 1: Imports & Load Checkpoint ────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/My Drive/IoT Intrusion Detection/data/'

df = pd.read_csv(DATA_PATH + 'UNSW_NB15_combined_raw.csv', low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: 2,540,047 rows × 49 columns ✓


In [ ]:
# ── Cell 2: Drop Stray Header Rows ───────────────────────────────────────────
# When we loaded with header=None, the 3 actual header rows in the raw files
# got read as data rows. We identify them by srcip == 'srcip' or containing BOM.

before = len(df)
df = df[~df['srcip'].astype(str).str.contains('srcip|ï»¿59.166', na=False)]
df = df.reset_index(drop=True)

print(f"Dropped {before - len(df)} stray header rows")
print(f"Remaining: {len(df):,} rows")

Dropped 1 stray header rows
Remaining: 2,540,046 rows


In [ ]:
# ── Debug: Find non-IP values in srcip ───────────────────────────────────────
# Show any rows where srcip doesn't look like an IP address
non_ip = df[~df['srcip'].astype(str).str.match(r'^\d+\.\d+\.\d+\.\d+$', na=False)]
print(f"Rows where srcip is not a standard IP: {len(non_ip)}")
print(non_ip['srcip'].value_counts().head(20))

Rows where srcip is not a standard IP: 0
Series([], Name: count, dtype: int64)


In [ ]:
# ── Cell 3: Drop Identifier & Leakage Columns ────────────────────────────────
# srcip, dstip  -> IP addresses don't generalise to unseen networks
# Stime, Ltime  -> raw timestamps, not useful as features
# attack_cat    -> names the attack type; would leak info in binary classification

DROP_COLS = ['srcip', 'dstip', 'Stime', 'Ltime', 'attack_cat']
df = df.drop(columns=DROP_COLS)

print(f"Remaining columns: {df.shape[1]}")
print(df.columns.tolist())

Remaining columns: 44
['sport', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'label']


In [ ]:
# ── Cell 4: Fix Mixed-Type Columns ───────────────────────────────────────────
# sport, dsport, ct_ftp_cmd loaded as object because some rows had strings.
# We coerce to numeric — anything that can't convert becomes NaN, then fill with 0.

mixed_cols = ['sport', 'dsport', 'ct_ftp_cmd']
for col in mixed_cols:
  df[col] = pd.to_numeric(df[col], errors='coerce')

print("Converted mixed-type columns to numeric")
print(df[mixed_cols].dtypes)

Converted mixed-type columns to numeric ✓
sport         float64
dsport        float64
ct_ftp_cmd    float64
dtype: object


In [ ]:
# ── Cell 5: Handle Nulls ──────────────────────────────────────────────────────
# is_ftp_login, ct_flw_http_mthd, ct_ftp_cmd -> null means protocol not active -> fill 0
# sport, dsport -> any remaining nulls after coercion -> fill 0

fill_zero_cols = ['is_ftp_login', 'ct_flw_http_mthd', 'ct_ftp_cmd',
                  'sport', 'dsport']
df[fill_zero_cols] = df[fill_zero_cols].fillna(0)

# Confirm no nulls remain
remaining_nulls = df.isnull().sum().sum()
print(f"Total nulls remaining: {remaining_nulls}")

Total nulls remaining: 0


In [ ]:
# ── Cell 6: Encode Categorical Columns ───────────────────────────────────────
# proto, state, service are nominal string columns — we label-encode them.
# LabelEncoder assigns an integer to each unique string value.

CAT_COLS = ['proto', 'state', 'service']
le = LabelEncoder()

for col in CAT_COLS:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"  {col}: {df[col].nunique()} unique values encoded")

print("\nCategorical encoding done")

  proto: 135 unique values encoded
  state: 16 unique values encoded
  service: 13 unique values encoded

Categorical encoding done ✓


In [ ]:
# ── Cell 7: Features & Label ──────────────────────────────────────────────────
X = df.drop(columns=['label'])
y = df['label']

print(f"Features (X): {X.shape}")
print(f"Label (y):    {y.shape}")
print(f"\nClass distribution:\n{y.value_counts()}")
print(f"Attack rate: {y.mean()*100:.1f}%")

Features (X): (2540046, 43)
Label (y):    (2540046,)

Class distribution:
label
0    2218763
1     321283
Name: count, dtype: int64
Attack rate: 12.6%


In [ ]:
# ── Cell 8: Train/Test Split ──────────────────────────────────────────────────
# stratify=y ensures both splits preserve the 12.6% attack ratio
# random_state=42 makes this reproducible

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} rows")
print(f"Test set:       {X_test.shape[0]:,} rows")
print(f"\nTrain attack rate: {y_train.mean()*100:.1f}%")
print(f"Test attack rate:  {y_test.mean()*100:.1f}%")

Training set:   2,032,036 rows
Test set:       508,010 rows

Train attack rate: 12.6%
Test attack rate:  12.6%


In [ ]:
# ── Cell 9: Scale Features ────────────────────────────────────────────────────
# StandardScaler transforms each feature to mean=0, std=1.
# We fit ONLY on training data, then transform both train and test.
# Fitting on test data too would be data leakage.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Features scaled")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

Features scaled ✓
X_train_scaled shape: (2032036, 43)
X_test_scaled shape:  (508010, 43)


In [ ]:
# ── Cell 10: Save Preprocessed Data ──────────────────────────────────────────
import joblib

# Save scaled arrays as numpy files
np.save(DATA_PATH + 'X_train_scaled.npy', X_train_scaled)
np.save(DATA_PATH + 'X_test_scaled.npy',  X_test_scaled)
np.save(DATA_PATH + 'y_train.npy',         y_train.values)
np.save(DATA_PATH + 'y_test.npy',          y_test.values)

# Save unscaled splits too (Random Forest doesn't need scaling)
X_train.to_csv(DATA_PATH + 'X_train.csv', index=False)
X_test.to_csv(DATA_PATH + 'X_test.csv',   index=False)
y_train.to_csv(DATA_PATH + 'y_train.csv', index=False)
y_test.to_csv(DATA_PATH + 'y_test.csv',   index=False)

# Save the scaler (needed to inverse-transform later if needed)
joblib.dump(scaler, DATA_PATH + 'scaler.pkl')

print("All preprocessed files saved to Drive")

All preprocessed files saved to Drive ✓
